# EDA: Airbnb in London Q4 2025

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats.mstats import winsorize
import plotly.graph_objects as go
from pyproj import Transformer
from sklearn.cluster import KMeans

In [2]:
df = pd.read_csv('../data/raw/listings.csv')
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.87,4.78,4.78,NaN,f,2,1,1,0,0.30
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.93,4.74,NaN,f,1,1,0,0,0.51
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.72,4.89,4.61,NaN,f,2,2,0,0,0.32
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.93,4.60,4.65,NaN,f,1,1,0,0,0.53
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,4.46,4.85,4.54,NaN,t,2,2,0,0,0.09


## Selecting features

In [3]:
df.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [4]:
drop_cols = ["listing_url", "scrape_id", "name",
                  "source","picture_url", "host_url", "host_id",
                "host_thumbnail_url", "host_picture_url", "estimated_occupancy_l365d", "estimated_revenue_l365d",
                 "host_location" ,"host_name", "neighbourhood_group_cleansed", "neighbourhood",
                  "availability_30", "availability_60", "availability_90", "availability_365", "availability_eoy",
                   "minimum_minimum_nights", "maximum_minimum_nights", "minimum_maximum_nights", "maximum_maximum_nights",
                   "minimum_nights_avg_ntm", "maximum_nights_avg_ntm", "calendar_updated", "calendar_last_scraped",
                   "has_availability","host_neighbourhood", "bathrooms_text",
                   "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly","reviews_per_month",
                   "license"]
df.drop(columns=drop_cols, inplace=True)

In [5]:
df.columns

Index(['id', 'last_scraped', 'description', 'neighborhood_overview',
       'host_since', 'host_about', 'host_response_time', 'host_response_rate',
       'host_acceptance_rate', 'host_is_superhost', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type',
       'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
       'amenities', 'price', 'minimum_nights', 'maximum_nights',
       'number_of_reviews', 'first_review', 'last_review',
       'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'instant_bookable',
       'calculated_host_listings_count',
       'calculated_host_listings_count_entire_homes',
       'calculated_host_listings_count_private_rooms',
       'calc

In [6]:
df.head()

,id,last_scraped,description,neighborhood_overview,host_since,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms
0,13913,2025-09-16,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,2009-11-16,I am a Multi-Media Visual Artist and Creative ...,within a few hours,100%,96%,t,...,4.80,4.81,4.87,4.78,4.78,f,2,1,1,0
1,15400,2025-09-16,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,2009-12-05,"English, grandmother, I have travelled quite ...",NaN,NaN,50%,f,...,4.87,4.88,4.84,4.93,4.74,f,1,1,0,0
2,17402,2025-09-16,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",2010-01-04,We are Liz and Jack. We manage a number of ho...,within an hour,88%,88%,t,...,4.72,4.72,4.72,4.89,4.61,f,2,2,0,0
3,24328,2025-09-18,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",2009-09-28,"I've been using Airbnb for a while now, both a...",within a few hours,100%,11%,f,...,4.91,4.90,4.93,4.60,4.65,f,1,1,0,0
4,36274,2025-09-15,*Update June '25- Pump Installed to improve wa...,NaN,2010-05-27,"We are Hendryks Services - your resident, mana...",within an hour,100%,98%,f,...,4.54,4.62,4.46,4.85,4.54,t,2,2,0,0


## Formating and creating features

In [7]:
df.isnull().sum()

id                                                  0
last_scraped                                        0
description                                      2450
neighborhood_overview                           55663
host_since                                         41
host_about                                      47038
host_response_time                              31707
host_response_rate                              31707
host_acceptance_rate                            27760
host_is_superhost                                1766
host_listings_count                                41
host_total_listings_count                          41
host_verifications                                 41
host_has_profile_pic                               41
host_identity_verified                             41
neighbourhood_cleansed                              0
latitude                                            0
longitude                                           0
property_type               

In [8]:
#removing missing values
#df.dropna(inplace=True)
df.shape

(96871, 43)

In [9]:
# converting data types to correct types
df["host_response_time"] = df["host_response_time"].astype("category")
#df["host_is_superhost"] = df["host_is_superhost"].map({"t": 1, "f": 0}).astype("int8")
#df["host_identity_verified"] = df["host_identity_verified"].map({"t": 1, "f": 0}).astype("int8")
df["last_scraped"] = pd.to_datetime(df["last_scraped"])
df["host_since"] = pd.to_datetime(df["host_since"])
df["host_response_rate"] = df["host_response_rate"].str.rstrip("%").astype("float") / 100
df["price"] = df["price"].str.replace("$", "").str.replace(",", "").astype("float")
df["first_review"] = pd.to_datetime(df["first_review"])
df["last_review"] = pd.to_datetime(df["last_review"])
df["instant_bookable"] = df["instant_bookable"].map({"t": 1, "f": 0}).astype("int8")
#df["host_has_profile_pic"] = df["host_has_profile_pic"].map({"t": 1, "f": 0}).astype("int8")
df["verifications"] = df["host_verifications"].apply(lambda x: len(x.strip("[]").split(", ")) if pd.notnull(x) else 0).astype("int8")
df["host_acceptance_rate"] = df["host_acceptance_rate"].str.rstrip("%").astype("float") / 100




In [10]:
df.head()

,id,last_scraped,description,neighborhood_overview,host_since,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,verifications
0,13913,2025-09-16,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,2009-11-16,I am a Multi-Media Visual Artist and Creative ...,within a few hours,1.00,0.96,t,...,4.81,4.87,4.78,4.78,0,2,1,1,0,2
1,15400,2025-09-16,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,2009-12-05,"English, grandmother, I have travelled quite ...",NaN,NaN,0.50,f,...,4.88,4.84,4.93,4.74,0,1,1,0,0,2
2,17402,2025-09-16,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",2010-01-04,We are Liz and Jack. We manage a number of ho...,within an hour,0.88,0.88,t,...,4.72,4.72,4.89,4.61,0,2,2,0,0,3
3,24328,2025-09-18,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",2009-09-28,"I've been using Airbnb for a while now, both a...",within a few hours,1.00,0.11,f,...,4.90,4.93,4.60,4.65,0,1,1,0,0,3
4,36274,2025-09-15,*Update June '25- Pump Installed to improve wa...,NaN,2010-05-27,"We are Hendryks Services - your resident, mana...",within an hour,1.00,0.98,f,...,4.62,4.46,4.85,4.54,1,2,2,0,0,2


## Feature engineering

### Host features 

In [11]:
df["experience"] = df["last_scraped"].dt.year - df["host_since"].dt.year

df["experience"]

0        16.0
1        16.0
2        15.0
3        16.0
4        15.0
         ... 
96866     6.0
96867     0.0
96868     4.0
96869     0.0
96870     6.0
Name: experience, Length: 96871, dtype: float64

In [12]:
df["about_dummy"] = df["host_about"].apply(lambda x: 0 if pd.isnull(x) else 1)

df["about_dummy"].value_counts()

about_dummy
1    49833
0    47038
Name: count, dtype: int64

In [13]:
df["host_response_time"].value_counts()

host_response_time
within an hour        43447
within a few hours    10683
within a day           6726
a few days or more     4308
Name: count, dtype: int64

In [14]:
df["cat_response_time"] = df["host_response_time"].apply(
    lambda x: 1 if x == "within an hour" 
    else 2 if x == "within a few hours" 
    else 3 if x == "within a day"
    else 4
)
df["cat_response_time"].value_counts()

cat_response_time
1    43447
4    36015
2    10683
3     6726
Name: count, dtype: int64

In [15]:
df["property_type"].value_counts()

property_type
Entire rental unit             41215
Private room in rental unit    14464
Private room in home           11704
Entire home                     9120
Entire condo                    8250
                               ...  
Lighthouse                         1
Private room in resort             1
Shared room in loft                1
Private room in barn               1
Shared room in guest suite         1
Name: count, Length: 91, dtype: int64

In [16]:
df["private"] = df["property_type"].apply(lambda x: 1 if "Private" in x else 0)
df["private"].value_counts()

private
0    64565
1    32306
Name: count, dtype: int64

In [17]:
df["entire"] = df["property_type"].apply(lambda x: 1 if "Entire" in x else 0)
df["entire"].value_counts()

entire
1    62626
0    34245
Name: count, dtype: int64

In [18]:
df["shared"] = df["property_type"].apply(lambda x: 1 if "Shared" in x else 0)
df["shared"].value_counts()

shared
0    96659
1      212
Name: count, dtype: int64

In [19]:
#converitng price to log price

df["log_price"] = np.log(df["price"])
df["win_log_p"] = winsorize(df["log_price"], limits=[0.01, 0.01])

In [20]:
#plotting property types
fig = px.histogram(df, x = "log_price", marginal="rug", nbins=200)


fig.show()

In [21]:


fig = go.Figure()

private_prices = df.loc[df["private"] == 1, "log_price"].dropna()
entire_prices  = df.loc[df["entire"] == 1, "log_price"].dropna()
shared_prices = df.loc[df["shared"] == 1, "log_price"].dropna()
other_prices = df.loc[(df["shared"] == 0) & (df["entire"] == 0) & (df["private"] == 0),
                       "log_price"].dropna()

fig.add_trace(go.Histogram(
    x=private_prices,
    nbinsx=50,
    histnorm="probability density",
    opacity=0.5,
    name="Private Room"
))

fig.add_trace(go.Histogram(
    x=entire_prices,
    nbinsx=50,
    histnorm="probability density",
    opacity=0.5,
    name="Entire Home"
))
fig.add_trace(go.Histogram(
    x=shared_prices,
    nbinsx=50,
    histnorm="probability density",
    opacity=0.5,
    name="Shared"
))

fig.add_trace(go.Histogram(
    x=other_prices,
    nbinsx=50,
    histnorm="probability density",
    opacity=0.7,
    name="Other"
))

fig.update_layout(
    barmode="overlay",
    title="Log Price by Room Type",
    xaxis_title= "Log Price"
)

fig.show()

## Creating fixed effects dummies

In [22]:
df.drop(columns=["last_scraped", "host_since", "description", "host_about",
                      "neighborhood_overview", "host_response_time", "property_type", 
                      "price"], inplace=True)

In [23]:
df.isnull().sum()

id                                                  0
host_response_rate                              31707
host_acceptance_rate                            27760
host_is_superhost                                1766
host_listings_count                                41
host_total_listings_count                          41
host_verifications                                 41
host_has_profile_pic                               41
host_identity_verified                             41
neighbourhood_cleansed                              0
latitude                                            0
longitude                                           0
room_type                                           0
accommodates                                        0
bathrooms                                       34846
bedrooms                                        12775
beds                                            34920
amenities                                           0
minimum_nights              

In [24]:
df.dropna(inplace=True)

In [25]:
df_sample = df.sample(10000, random_state=42)

fig = px.scatter_mapbox(
    df_sample,
    lat="latitude",
    lon="longitude",
    zoom=10,
    height=600,
)

fig.update_traces(marker=dict(size=4), opacity=0.25)

fig.update_layout(
    mapbox_style="carto-positron",
    title="Geographic Distribution of Airbnb Listings (London)",
    margin=dict(r=0, t=50, l=0, b=0)
)

fig.show()

C:\Users\danil\AppData\Local\Temp\ipykernel_30252\977692311.py:3: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [26]:
df_sample = df.sample(10000, random_state=42)

fig = px.scatter_mapbox(
    df_sample,
    lat="latitude",
    lon="longitude",
    color="log_price",
    color_continuous_scale="Viridis",
    zoom=10,
    height=600,
)

fig.update_traces(marker=dict(size=4), opacity=0.5)

fig.update_layout(
    mapbox_style="carto-positron",
    title="Spatial Distribution of Log Prices",
    margin=dict(r=0, t=50, l=0, b=0)
)

fig.show()

C:\Users\danil\AppData\Local\Temp\ipykernel_30252\485455429.py:3: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



### K-means clustering 

In [27]:
# WGS84 → British National Grid (EPSG:27700)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)

x, y = transformer.transform(df["longitude"].values,
                             df["latitude"].values)

df["x"] = x
df["y"] = y

In [28]:
fig = px.scatter(df, x = "x", y = "y")
fig.show()

In [29]:
K = 100  

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)

df["micro_loc_fe_100"] = kmeans.fit_predict(df[["x", "y"]])

In [30]:
fig = px.scatter(df, x = "x", y = "y", color="micro_loc_fe_100")
fig.update_layout(
    title = "FE variable based on location in Euclidien space"
)
fig.show()

In [31]:
fig = px.scatter(df, x = "latitude", y = "longitude", color="micro_loc_fe_100")
fig.update_layout(
    title = "FE variable based on location in Cordinate space"
)
fig.show()

In [32]:
cluster_sizes = df["micro_loc_fe_100"].value_counts()

cluster_sizes.describe()

count     100.000000
mean      428.980000
std       356.227512
min        33.000000
25%       158.250000
50%       328.000000
75%       584.250000
max      1498.000000
Name: count, dtype: float64

In [33]:
K = 150 

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)

df["micro_loc_fe_150"] = kmeans.fit_predict(df[["x", "y"]])

In [34]:
fig = px.histogram(cluster_sizes, nbins=25)
fig.update_layout(
    title = "Distribution of number of listings inside FE"
)
fig.show()

In [36]:
fig = go.Figure()

sizes_100 = df["micro_loc_fe_100"].value_counts()
sizes_150 = df["micro_loc_fe_150"].value_counts()

fig.add_trace(go.Histogram(
    x=sizes_100.values,
    nbinsx=50,
    histnorm="probability density",
    opacity=0.5,
    name="K = 100"
))

fig.add_trace(go.Histogram(
    x=sizes_150.values,
    nbinsx=50,
    histnorm="probability density",
    opacity=0.5,
    name="k = 150"
))
fig.update_layout(
    barmode="overlay",
    title="Distribution of number of listings inside FE",
    xaxis_title= "Count"
)

fig.show()

## Treatment variable